In [42]:
import pandas as pd
import nfl_data_py as nfl

In [47]:
from data.data_aggregation import import_pbp_data
from simulation.drive_selection import select_drive
from simulation.overtime_period import Overtime_Period

In [ ]:
select_drive()

In [53]:
DATA_DIR = r"C://Users//natel//PycharmProjects//NFL_Overtime_Model//data"
df = pd.read_pickle(f'{DATA_DIR}//pbp_data.pkl')
drive_list = pd.read_csv(f'{DATA_DIR}//drive_list.csv')
ko_list = pd.read_csv(f"{DATA_DIR}//ko_list.csv")


In [66]:
agg_results = {}

for season in [2015, 2016,2023, 2024, 2025]:
    first_drive_results = []
    for _ in range(1000):
        OT = Overtime_Period('Kicking Team', 'Receiving Team', 'Kicking Team', season)  # was hardcoded to 2024
        OT.simulate()
        lines = OT.summary.split('\n')
        for line in lines:
            if ('scores a touchdown' in line or 'kicks a field goal' in line or
                'turns the ball over' in line or 'punts' in line):
                first_drive_results.append(line.strip())
                break
    agg_results[season] = pd.Series(first_drive_results).value_counts(normalize=True)

agg_df = pd.DataFrame(agg_results).fillna(0)
display(agg_df)

,2015,2016,2023,2024,2025
Kicking Team scores a touchdown.,0.000000,0.000000,0.000000,0.000000,0.002002
Receiving Team kicks a field goal.,0.276104,0.213925,0.192192,0.148445,0.170170
Receiving Team punts.,0.420683,0.240161,0.459459,0.290873,0.379379
Receiving Team scores a touchdown.,0.055221,0.271443,0.123123,0.400201,0.274274
Receiving Team turns the ball over via DOWNS.,0.007028,0.087790,0.006006,0.060181,0.029029
Receiving Team turns the ball over via FUMBLE.,0.139558,0.030272,0.058058,0.062187,0.044044
Receiving Team turns the ball over via INTERCEPTION.,0.040161,0.072654,0.102102,0.024072,0.051051
Receiving Team turns the ball over via INTERCEPTION.Kicking Team returns it for a touchdown.,0.000000,0.000000,0.000000,0.000000,0.001001
Receiving Team turns the ball over via MISSED_FG.,0.061245,0.083754,0.059059,0.014042,0.049049


In [60]:
agg_df

,2011,2015,2016,2024,2025
Kicking Team scores a touchdown.,0.000000,0.000000,0.000000,0.000000,0.003003
Kicking Team turns the ball over via DOWNS.,0.000000,0.000000,0.000000,0.000000,0.001001
Receiving Team kicks a field goal.,0.291751,0.272362,0.216650,0.142570,0.163163
Receiving Team punts.,0.420523,0.458291,0.239719,0.314257,0.342342
Receiving Team scores a touchdown.,0.071429,0.047236,0.261785,0.390562,0.314314
Receiving Team turns the ball over via BLOCKED_FG.,0.001006,0.001005,0.002006,0.000000,0.000000
Receiving Team turns the ball over via DOWNS.,0.011066,0.010050,0.107322,0.056225,0.024024
Receiving Team turns the ball over via FUMBLE.,0.125755,0.120603,0.023069,0.064257,0.044044
Receiving Team turns the ball over via INTERCEPTION.,0.031187,0.032161,0.060181,0.020080,0.057057
Receiving Team turns the ball over via MISSED_FG.,0.047284,0.058291,0.089268,0.012048,0.051051


In [65]:
ko_list.groupby('season')['starting_field_position'].mean()*-1+100

season
2006    28.136564
2007    28.975203
2008    28.215046
2009    27.354331
2010    27.856379
2011    23.117691
2012    23.186805
2013    23.240304
2014    22.933755
2015    22.614872
2016    25.698535
2017    25.496012
2018    25.965326
2019    25.858473
2020    26.048302
2021    25.670486
2022    26.063966
2023    25.776596
2024    30.336301
2025    31.133264
Name: starting_field_position, dtype: float64

In [50]:
drive_list['season'] = drive_list['drive_id'].str[:4].astype(int)
print(drive_list.groupby('season')['drive_result']
      .apply(lambda x: (x == 'TOUCHDOWN').mean())
      .round(3))


season
2006    0.181
2007    0.192
2008    0.195
2009    0.196
2010    0.197
2011    0.192
2012    0.198
2013    0.200
2014    0.203
2015    0.202
2016    0.213
2017    0.194
2018    0.225
2019    0.219
2020    0.253
2021    0.231
2022    0.213
2023    0.206
2024    0.229
2025    0.231
Name: drive_result, dtype: float64


In [33]:
tied_drives = drive_list[drive_list['start_score_diff'] == 0]
print(tied_drives['drive_result'].value_counts(normalize=True))

drive_result
PUNT                    0.444202
TOUCHDOWN               0.205997
FIELD_GOAL              0.164448
INTERCEPTION            0.064898
FUMBLE                  0.043294
MISSED_FG               0.028892
DOWNS                   0.021603
END_HALF                0.020076
SAFETY                  0.002226
BLOCKED_FG              0.001702
BLOCKED_PUNT            0.001528
END_GAME                0.000349
FUMBLE,_SAFETY          0.000349
BLOCKED_PUNT,_DOWNS     0.000262
BLOCKED_FG,_DOWNS       0.000131
BLOCKED_PUNT,_SAFETY    0.000044
Name: proportion, dtype: float64


In [12]:
result = select_drive(drive_list, 25, 900, 0, 2011)
print(result)

drive_id                     2009_17_TEN_SEA_17
start_yardline                               31
start_time_left                           925.0
start_posteam_score                        10.0
start_defteam_score                        10.0
start_score_diff                            0.0
drive_result                         FIELD_GOAL
time_elapsed                              173.0
defteam_TD                                False
posteam_score_change                        3.0
defteam_score_change                        0.0
last_play_yardline                          2.0
last_ydstogo                                2.0
next_drive_start_yardline                  70.0
Name: 24476, dtype: object


In [7]:
import pandas as pd

def get_ruleset(season, week):
    is_playoff = week > 18
    if season >= 2025:
        return 'post2025'
    elif season >= 2022 and is_playoff:
        return 'post2025'  # same rules as post2025
    elif season >= 2012:
        return '2012_2024'
    else:
        return 'pre2012'

def aggregate_overtime_games(pbp: pd.DataFrame) -> pd.DataFrame:
    ot_plays = pbp[pbp['game_half'] == 'Overtime'].copy()

    results = []

    for game_id, game in ot_plays.groupby('game_id'):
        game = game.sort_values('play_id')

        first_play = game.iloc[0]
        season = first_play['season']
        week = first_play['week']
        receiving_team = first_play['posteam']

        home = first_play['home_team']
        away = first_play['away_team']
        kicking_team = away if receiving_team == home else home

        last_play = game.iloc[-1]
        home_score = last_play['total_home_score']
        away_score = last_play['total_away_score']

        if home_score > away_score:
            winner = home
        elif away_score > home_score:
            winner = away
        else:
            winner = 'TIE'

        results.append({
            'game_id': game_id,
            'season': season,
            'week': week,
            'ruleset': get_ruleset(season, week),
            'kicking_team': kicking_team,
            'receiving_team': receiving_team,
            'winner': winner,
            'kicking_team_won': winner == kicking_team,
            'receiving_team_won': winner == receiving_team,
            'tie': winner == 'TIE',
        })

    return pd.DataFrame(results).sort_values(['season', 'week']).reset_index(drop=True)

In [8]:
ot_df = aggregate_overtime_games(df)

In [9]:
ot_df.groupby('ruleset').agg(
    games=('game_id', 'count'),
    receiving_won=('receiving_team_won', 'sum'),
    kicking_won=('kicking_team_won', 'sum'),
    ties=('tie', 'sum'),
    receiving_pct=('receiving_team_won', 'mean'),
    kicking_pct=('kicking_team_won', 'mean'),
    tie_pct=('tie', 'mean'),
).round(3).T


ruleset,2012_2024,post2025,pre2012
games,211.000,17.000,93.000
receiving_won,113.000,9.000,49.000
kicking_won,83.000,7.000,43.000
ties,15.000,1.000,1.000
receiving_pct,0.536,0.529,0.527
kicking_pct,0.393,0.412,0.462
tie_pct,0.071,0.059,0.011
